# Iris Probabilistic Generative Classifier
This notebook progressive explains the manual implementation of a Gaussian generative classifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

## Load and Inspect Dataset

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['target'] = [target_names[i] for i in y]
df.head()

## Class Distribution

In [ ]:
sns.countplot(x='target', data=df)
plt.title('Class Distribution')
plt.show()

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## Calculate Class Priors P(C)

In [ ]:
classes = np.unique(y_train)
N = X_train.shape[0]
priors = {}

for c in classes:
    N_c = X_train[y_train == c].shape[0]
    priors[c] = N_c / N
    print(f'Prior for {target_names[c]}: {priors[c]:.4f}')

## Calculate Mean Vectors $\mu_k$

In [ ]:
means = {}
for c in classes:
    X_c = X_train[y_train == c]
    means[c] = np.mean(X_c, axis=0)
    print(f'Mean for {target_names[c]}: {means[c]}')

## Calculate Covariance Matrices $\Sigma_k$
Using Maximum Likelihood Estimation ($1/N_k$)

In [ ]:
covariances = {}
for c in classes:
    X_c = X_train[y_train == c]
    N_c = X_c.shape[0]
    X_c_centered = X_c - means[c]
    covariances[c] = (X_c_centered.T @ X_c_centered) / N_c
    print(f'Covariance for {target_names[c]}:\n{covariances[c]}\n')

## Gaussian Density and Likelihoods P(x|C)

In [ ]:
def log_gaussian_density(x, mean, cov):
    d = len(mean)
    sign, logdet = np.linalg.slogdet(cov)
    x_centered = x - mean
    inv_cov_x = np.linalg.solve(cov, x_centered.T)
    mahalanobis = np.sum(x_centered * inv_cov_x.T, axis=1)
    return -0.5 * d * np.log(2 * np.pi) - 0.5 * logdet - 0.5 * mahalanobis

# Example flower
x_example = np.array([[6.0, 2.9, 4.5, 1.5]])
for c in classes:
    log_lik = log_gaussian_density(x_example, means[c], covariances[c])[0]
    print(f'Likelihood for {target_names[c]}: {np.exp(log_lik):.4e}')

## Apply Bayes Theorem for Posterior Probabilities P(C|x)

In [ ]:
joint_probs = []
for c in classes:
    log_lik = log_gaussian_density(x_example, means[c], covariances[c])[0]
    joint = np.exp(log_lik) * priors[c]
    joint_probs.append(joint)

posteriors = joint_probs / np.sum(joint_probs)
for c in classes:
    print(f'Posterior for {target_names[c]}: {posteriors[c]:.4f}')

## Predict Unknown Flower

In [ ]:
pred_idx = np.argmax(posteriors)
print(f'Predicted class: {target_names[pred_idx]}')

## Evaluate Classifier (using our src package)

In [ ]:
import sys
sys.path.append('..')
from src.generative_classifier import GaussianGenerativeClassifier

model = GaussianGenerativeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

## Show Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.show()

## Conclusion
We successfully implemented a Gaussian generative classifier from scratch.